# Week 3 — V1 Data Pipeline: JalanJalanMY

**Project:** JalanJalanMY — grounded itinerary planner for Malaysia
**Pathway:** A (PDF/TXT) — the knowledge source is official tourism documents
**Sources:** `IDB_PDF_KLD_EN.pdf` (S01, prepared) and `Map_Guide_PENANG_Eng.pdf` (S02, inspected only) — Tourism Malaysia guides

**Run order:** Cell 1 → upload PDFs → Cell 3 → A3 inspection → Cell 4 → Cell 4A → Cell 5A1 → B3 comparison → Cell 5A2 → B3 output check → B4.1 register → Cell 6A → Cell 6B → Cell 7 → Cell 8A → Cell 9 → C4 download.

> The source PDFs are © Tourism Malaysia (all rights reserved). They are used here for coursework only and are **not** committed to the public repository.

## Part A — Start with the data

### A1 — Requirements that need this data
| Week 2 requirement | Data it needs | Source here |
|---|---|---|
| **FR3 / RAI1** — suggest only attractions from official sources, with a source reference | Official attraction names, descriptions and locations | Tourism Malaysia guides (S01, S02) |
| **NFR2 / RAI4** — correct hours and fees, with a "last checked" date | Opening hours, fees, publication date | Hours/fees in the guides (checked in A3); PDF metadata dates |
| **FR4** — feasibility checks | Opening hours and locations per attraction | Partly available — see A3 limitation |

In [1]:
# Cell 1 — Workspace setup (RUN FIRST)
from pathlib import Path

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility aliases for notebook examples
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)


Raw folder: /content/week3_project/data/raw
Processed folder: /content/week3_project/data/processed


### Upload the source PDFs
Upload the two PDFs anywhere in the Colab Files panel (e.g. the top `/content` folder) or straight into `week3_project/data/raw`, then run the cell below. It moves them into `data/raw`, or opens an upload box if they are missing.

In [2]:
# Upload helper — moves PDFs uploaded to /content into data/raw, then checks
import shutil
from pathlib import Path
needed = ["IDB_PDF_KLD_EN.pdf", "Map_Guide_PENANG_Eng.pdf"]
for name in needed:
    loose = Path("/content") / name
    if loose.is_file() and not (RAW_DIR / name).is_file():
        shutil.move(str(loose), str(RAW_DIR / name))
        print("Moved to data/raw:", name)
present = [n for n in needed if (RAW_DIR / n).is_file()]
if len(present) == len(needed):
    print("Already in data/raw:", present)
else:
    try:
        from google.colab import files
        uploaded = files.upload()
        for name, data in uploaded.items():
            (RAW_DIR / name).write_bytes(data)
            print("Saved to data/raw:", name)
    except ImportError:
        print("Not running in Colab — copy the PDFs into", RAW_DIR)


Moved to data/raw: IDB_PDF_KLD_EN.pdf
Moved to data/raw: Map_Guide_PENANG_Eng.pdf
Already in data/raw: ['IDB_PDF_KLD_EN.pdf', 'Map_Guide_PENANG_Eng.pdf']


In [3]:
# Cell 3 — Verify uploaded inputs (self-contained)
from pathlib import Path
RAW = Path("/content/week3_project/data/raw")
RAW_DIR = RAW
BASE_DIR = RAW.parent.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
BASE = BASE_DIR
OUT = PROCESSED_DIR

files = sorted(f for f in RAW.rglob("*") if f.is_file())
print("Input count:", len(files))
for f in files:
     print(f.relative_to(RAW), "bytes:", f.stat().st_size)

assert files, "Upload at least one relevant input file before proceeding."
print("Upload verified. Select your relevant input(s) for Step A3.")


Input count: 2
IDB_PDF_KLD_EN.pdf bytes: 6122024
Map_Guide_PENANG_Eng.pdf bytes: 4178106
Upload verified. Select your relevant input(s) for Step A3.


### A3 — Inspect the selected sources
Opens both PDFs and shows the actual extracted content, not just that the files exist.

In [4]:
# A3 inspection — S01 and S02
!pip -q install pypdf
import re
from pypdf import PdfReader

def inspect_pdf(name):
    reader = PdfReader(str(RAW_DIR / name))
    pages = [p.extract_text() or "" for p in reader.pages]
    text = "\n".join(pages)
    lines = [l for l in text.splitlines() if l.strip()]
    meta = reader.metadata or {}
    print("=" * 70)
    print("File:", name)
    print("Pages:", len(pages), "| empty pages:", sum(1 for p in pages if not p.strip()))
    print("Extracted characters:", len(text), "| non-empty lines:", len(lines))
    print("Very short lines (<= 4 chars, e.g. map labels):", sum(1 for l in lines if len(l.strip()) <= 4))
    print("PDF dates:", meta.get("/CreationDate"), "|", meta.get("/ModDate"))
    print("Mentions — opening hours/times:", len(re.findall(r"(?i)opening hours|\d{1,2}[.:]\d{2}\s?(?:am|pm)", text)),
          "| RM prices:", len(re.findall(r"RM ?\d", text)),
          "| phone numbers:", len(re.findall(r"\+60\d", text)))
    print("Sample 1 (start):", repr(text[:300]))
    k = len(text) // 2
    print("Sample 2 (middle):", repr(text[k:k + 300]))
    return text

s01_text = inspect_pdf("IDB_PDF_KLD_EN.pdf")
s02_text = inspect_pdf("Map_Guide_PENANG_Eng.pdf")


File: IDB_PDF_KLD_EN.pdf
Pages: 72 | empty pages: 0
Extracted characters: 143570 | non-empty lines: 3683
Very short lines (<= 4 chars, e.g. map labels): 303
PDF dates: None | D:20250318100828Z
Mentions — opening hours/times: 8 | RM prices: 1 | phone numbers: 16
Sample 1 (start): '1\nThe Dazzling Capital City\nKUALA LUMPUR\nPetronas Twin Towers\n2\n4 DOING THE SIGHTS \n5 Prestigious Landmarks\n6 Heritage Sites\n10 Places of Worship\n12  Themed Attractions\n14 TROPICAL ENCLAVES\n15 Perdana Botanical Gardens\n16 KLCC Park\n17 Titiwangsa Lake Gardens\n17  KL Forest Eco Park\n17 Bukit Nanas For'
Sample 2 (middle): ' specialty during Hari Raya. \nIt is known for its hot and spicy taste. \nRestaurants serving Continental, Vietnamese, \nKorean, Japanese, Thai, and Indonesian cuisine can \nbe found at major hotels and in the vicinity of Jalan \nSultan Ismail, Jalan Ampang, Sri Hartamas, Mont \nKiara and Bangsar. The Ain'
File: Map_Guide_PENANG_Eng.pdf
Pages: 2 | empty pages: 0
Extracted charact

### A3 findings
| | S01 — KL guide | S02 — Penang Map & Guide |
|---|---|---|
| **Structure** | 72-page brochure, sections (landmarks, heritage, parks, museums…) | 2-page fold-out map with numbered attraction descriptions |
| **Extraction** | Readable prose, no empty pages | Descriptions readable, but mixed with map-label fragments (street names, grid numbers, "To", single words) |
| **Useful fields** | Attraction names, addresses, descriptions, websites | Names, descriptions, phone numbers |
| **Hours / fees** | Very few (8 time mentions, 1 RM price) | Very few (5 time mentions) |
| **Date** | PDF modified March 2025 | PDF modified March 2024 |

**Ingestion limitation:** the guides support **FR3** (official attraction list) but **not** NFR2/FR4 on their own, because opening hours and fees are mostly missing. A second source of hours is needed (recorded in the data inventory). **S01 is selected for preparation** because it extracts cleanly; S02 is kept as inspection evidence of the map-layout problem.

## Part B — Prepare the data

In [5]:
# Cell 4 — REQUIRED: Source Selection + Lineage Utilities
# Run ONCE after Cell 1 and Cell 3

from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

# Select the exact source before parsing.
# Never automatically take the first file by extension.

def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(
                f"Source not found: {selected}"
            )
        return selected
    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )
    print(
        "Candidates:",
        [str(p.relative_to(RAW_DIR)) for p in matches]
    )
    if len(matches) != 1:
        raise ValueError(
            "Select exactly one source. "
            "Set relative_path to an exact file under RAW_DIR."
        )
    return matches[0]
def source_hash(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()
def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )
    print("Saved lineage:", target)
    return manifest

# Confirmation messages
print("Cell 4 completed successfully.")
print("Source selection and lineage utilities are ready.")
print("Available functions:")
print("- select_source()")
print("- source_hash()")
print("- save_manifest()")


Cell 4 completed successfully.
Source selection and lineage utilities are ready.
Available functions:
- select_source()
- source_hash()
- save_manifest()


In [6]:
# CELL 4A — COMMON LINEAGE REGISTRY
# REQUIRED: Run once before selected Part B pathways
from pathlib import Path
lineage_entries = {}
def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()
    # Validate source
    if not source.is_file():
        raise FileNotFoundError(
            f"Source file not found: {source}"
        )
    # Validate prepared output
    if not output.is_file():
        raise FileNotFoundError(
            f"Prepared output not found: {output}"
        )
    if output.stat().st_size == 0:
        raise ValueError(
            f"Prepared output is empty: {output}"
        )
    # Validate processing description
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError(
            "A processing description is required."
        )
    processing = processing.strip()
    if processing.upper().startswith("REPLACE"):
        raise ValueError(
            "Replace the placeholder with the actual method."
        )
    # Ensure source and output belong to project workspace
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())
    # Same source/output pair is updated, not duplicated
    key = (str(source), str(output))
    lineage_entries[key] = {
        "source": source,
        "output": output,
        "processing": processing
    }
    print(
        "Registered:",
        source.name,
        "->",
        output.name
    )
print("Common lineage registry ready.")
print("Registered entries:", len(lineage_entries))


Common lineage registry ready.
Registered entries: 0


In [7]:
# Cell 5A1
# PATHWAY A1 — PDF/TXT
# CONDITIONAL REQUIRED: Run only for document-based projects
# Install PDF processing library
!pip -q install pypdf

from pypdf import PdfReader
import re

# Step 1: Select the exact source file
# JalanJalanMY source S01 (see docs/week3_data_inventory.md)
f = select_source(
    relative_path="IDB_PDF_KLD_EN.pdf"  # JalanJalanMY: Tourism Malaysia KL guide
)

# Step 2: Extract text according to file type
if f.suffix.lower() == ".pdf":
    reader = PdfReader(str(f))
    pages = [
        (i + 1, page.extract_text() or "")
        for i, page in enumerate(reader.pages)
    ]
elif f.suffix.lower() == ".txt":
    pages = [(1, f.read_text(encoding="utf-8"))]
else:
    raise ValueError("Pathway A needs a PDF or TXT source")
# Step 3: Basic text cleaning
text = "\n".join(content for _, content in pages)
clean = re.sub(r"[ \t]+", " ", text).strip()

print("Source:", f.name)
print("Pages/segments:", len(pages))
print("Extracted characters:", len(text))
print("Cleaned characters:", len(clean))
print("Content sample:", repr(clean[:300]))

assert clean, "No usable text: check extraction or OCR needs."
# Step 4: Fixed-size chunking
def fixed_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    return [
        s[i:i + size]
        for i in range(0, len(s), size)
    ]
# Step 5: Paragraph-aware chunking
def paragraph_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    parts = []
    current = ""
    for segment in re.split(r"\n\s*\n", s):
        segment = segment.strip()
        if not segment:
            continue
        if len(segment) > size:
            if current:
                parts.append(current)
                current = ""
            parts.extend(fixed_chunks(segment, size))

        elif len(current) + len(segment) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + segment
        else:
            if current:
                parts.append(current)
            current = segment
    if current:
        parts.append(current)
    return parts
# Step 6: Compare two preparation approaches
fixed = fixed_chunks(clean, size=500)
structured = paragraph_chunks(clean, size=500)

print("\n--- Chunking Comparison ---")
print("Fixed chunks:", len(fixed))
print("Paragraph-aware chunks:", len(structured))

print("\nFixed sample:")
print(repr(fixed[0][:180]))

print("\nParagraph-aware sample:")
print(repr(structured[0][:180]))

print("\nPATHWAY A1 completed successfully.")


Source: IDB_PDF_KLD_EN.pdf
Pages/segments: 72
Extracted characters: 143570
Cleaned characters: 143238
Content sample: '1\nThe Dazzling Capital City\nKUALA LUMPUR\nPetronas Twin Towers\n2\n4 DOING THE SIGHTS \n5 Prestigious Landmarks\n6 Heritage Sites\n10 Places of Worship\n12 Themed Attractions\n14 TROPICAL ENCLAVES\n15 Perdana Botanical Gardens\n16 KLCC Park\n17 Titiwangsa Lake Gardens\n17 KL Forest Eco Park\n17 Bukit Nanas Fores'

--- Chunking Comparison ---
Fixed chunks: 287
Paragraph-aware chunks: 303

Fixed sample:
'1\nThe Dazzling Capital City\nKUALA LUMPUR\nPetronas Twin Towers\n2\n4 DOING THE SIGHTS \n5 Prestigious Landmarks\n6 Heritage Sites\n10 Places of Worship\n12 Themed Attractions\n14 TROPICAL '

Paragraph-aware sample:
'1\nThe Dazzling Capital City\nKUALA LUMPUR\nPetronas Twin Towers\n2\n4 DOING THE SIGHTS \n5 Prestigious Landmarks\n6 Heritage Sites\n10 Places of Worship\n12 Themed Attractions\n14 TROPICAL '

PATHWAY A1 completed successfully.


### B3 — Compare the two chunking approaches
Same input, two configurations. Extra measurements beyond the chunk counts:

In [8]:
# B3 comparison evidence (same cleaned text, both 500-character configurations)
segments = [s for s in re.split(r"\n\s*\n", clean) if s.strip()]
print("Blank-line segments in source:", len(segments),
      "| longer than 500 chars:", sum(len(s) > 500 for s in segments),
      "| longest:", max(len(s) for s in segments))

def chunk_stats(name, chunks):
    lengths = [len(c) for c in chunks]
    mid_word = sum(1 for a, b in zip(chunks, chunks[1:]) if a[-1:].isalpha() and b[:1].isalpha())
    sentence_end = sum(1 for c in chunks if c.rstrip()[-1:] in '.!?)"\u201d')
    print(f"{name:16} chunks={len(chunks):4}  min={min(lengths):3}  max={max(lengths)}  "
          f"cut mid-word={mid_word:4}  end at sentence={sentence_end}")

chunk_stats("fixed", fixed)
chunk_stats("paragraph-aware", structured)

# Representative chunk: the same attraction in both outputs
for label, chunks in (("fixed", fixed), ("paragraph-aware", structured)):
    i = next(i for i, c in enumerate(chunks) if "KL Tower" in c)
    print(f"\n{label} chunk {i}:", repr(chunks[i][:220]))


Blank-line segments in source: 29 | longer than 500 chars: 29 | longest: 37080
fixed            chunks= 287  min=238  max=500  cut mid-word= 163  end at sentence=6
paragraph-aware  chunks= 303  min= 14  max=500  cut mid-word= 135  end at sentence=22

fixed chunk 7: 'th attractions that will surely make your visit a \nbreathtaking experience.\nwww.merdeka-tower.com\nKL Tower\nJalan Puncak, Oﬀ Jalan P.Ramlee\nKL Tower is one of the most prominent and \ndistinctive landmarks in the country. '

paragraph-aware chunk 7: 'th attractions that will surely make your visit a \nbreathtaking experience.\nwww.merdeka-tower.com\nKL Tower\nJalan Puncak, Oﬀ Jalan P.Ramlee\nKL Tower is one of the most prominent and \ndistinctive landmarks in the country. '


### B3 decision
| Configuration A | Configuration B | Observed evidence | Selected approach and trade-off |
|---|---|---|---|
| Fixed-size, 500 characters | Paragraph-aware, 500 characters | Fixed: 287 chunks, 159 cut mid-word, 6 end at a sentence. Paragraph-aware: 303 chunks, 133 cut mid-word, 22 end at a sentence. All 29 blank-line segments exceed 500 characters, so both methods slice inside attraction entries (e.g. the "KL Tower" chunk starts mid-word). | **Paragraph-aware.** Benefit: respects the guide's section breaks and gives fewer broken words, which helps FR3/RAI1 (attraction text kept together for source traceability). Accepted limitation: the PDF has few blank lines, so most chunks are still 500-character slices that can split one attraction across two chunks, and page numbers are not kept (`page: None`). Splitting on attraction headings is a later improvement. |

In [9]:
# Cell 5A2
# PATHWAY A2 — REQUIRED after A1
# Choose and export traceable text chunks
import json
# Step 1: Select the preparation approach
# Change to "fixed" if your Part A1 evidence supports it.
CHOSEN_METHOD = "paragraph-aware"
if CHOSEN_METHOD == "paragraph-aware":
    chosen = structured
    approach = "paragraph-aware-char-500"
elif CHOSEN_METHOD == "fixed":
    chosen = fixed
    approach = "fixed-char-500"
else:
    raise ValueError("Choose 'paragraph-aware' or 'fixed'")
# Step 2: Create traceable records
records = [
    {
        "record_id": f"{f.stem}-{i:04d}",
        "text": chunk,
        "source": str(f.relative_to(RAW_DIR)),
        "source_sha256": source_hash(f),
        "processing": approach,
        "page": None
    }
    for i, chunk in enumerate(chosen, 1)
]
# Page is None because the basic whole-document
# splitter does not preserve page boundaries.
# Step 3: Export prepared records
prepared_path = PROCESSED_DIR / "prepared_text.jsonl"
prepared_path.parent.mkdir(parents=True, exist_ok=True)
with prepared_path.open("w", encoding="utf-8") as output:
    for row in records:
        output.write(
            json.dumps(row, ensure_ascii=False) + "\n"
        )
# Step 4: Basic validation
assert prepared_path.is_file(), "Prepared output not found"
assert prepared_path.stat().st_size > 0, "Prepared output is empty"
assert records, "No text chunks were produced"
assert all(row["text"].strip() for row in records), \
    "One or more chunks are empty"
with prepared_path.open("r", encoding="utf-8") as file:
    exported_records = [
        json.loads(line)
        for line in file
        if line.strip()
    ]
assert len(exported_records) == len(records), \
    "Exported record count does not match"

# Step 5: Display evidence
print("PATHWAY A2 completed successfully.")
print("Selected approach:", approach)
print("Total chunks exported:", len(records))
print("Prepared file:", prepared_path)
print("File size:", prepared_path.stat().st_size, "bytes")
print("Validation: PASS")
print("First record:", records[0])


PATHWAY A2 completed successfully.
Selected approach: paragraph-aware-char-500
Total chunks exported: 303
Prepared file: /content/week3_project/data/processed/prepared_text.jsonl
File size: 214828 bytes
Validation: PASS
First record: {'record_id': 'IDB_PDF_KLD_EN-0001', 'text': '1\nThe Dazzling Capital City\nKUALA LUMPUR\nPetronas Twin Towers\n2\n4 DOING THE SIGHTS \n5 Prestigious Landmarks\n6 Heritage Sites\n10 Places of Worship\n12 Themed Attractions\n14 TROPICAL ENCLAVES\n15 Perdana Botanical Gardens\n16 KLCC Park\n17 Titiwangsa Lake Gardens\n17 KL Forest Eco Park\n17 Bukit Nanas Forest Reserve\n18 TREASURE TROVES \n19 Museums\n21 Galleries\n22 Memorials\n23 RELAX AND REJUVENATE \n24 Spa Retreats\n25 Healthcare\n26 ENTHRALLING PERFORMANCES \n27 Premier Concert Halls\n27 Performances\n28 Fine ', 'source': 'IDB_PDF_KLD_EN.pdf', 'source_sha256': '1484ba6f1b3203f2f1f3ec96c86f8bfcc6bf0ec6988610d0b4fe207eb65b76f7', 'processing': 'paragraph-aware-char-500', 'page': None}


### B3 — Basic output check (required)

In [10]:
assert prepared_path.exists(), 'Prepared output missing.'
assert prepared_path.stat().st_size > 0, 'Prepared output is empty.'
print('Output:',prepared_path, 'Size:',prepared_path.stat().st_size)
print('Validation: file exists and is non-empty; verify content manually too.')

# Manual content check: two representative prepared records
import json
with prepared_path.open(encoding="utf-8") as fh:
    rows = [json.loads(line) for line in fh if line.strip()]
for row in (rows[7], rows[len(rows) // 2]):
    print("\n", row["record_id"], "|", row["processing"], "|", repr(row["text"][:200]))


Output: /content/week3_project/data/processed/prepared_text.jsonl Size: 214828
Validation: file exists and is non-empty; verify content manually too.

 IDB_PDF_KLD_EN-0008 | paragraph-aware-char-500 | 'th attractions that will surely make your visit a \nbreathtaking experience.\nwww.merdeka-tower.com\nKL Tower\nJalan Puncak, Oﬀ Jalan P.Ramlee\nKL Tower is one of the most prominent and \ndistinctive landma'

 IDB_PDF_KLD_EN-0152 | paragraph-aware-char-500 | ' Hartamas, Mont \nKiara and Bangsar. The Ain Al Arabia or Arab Square \nin Bukit Bintang oﬀers a host of culinary fare from \nthe Middle East.\nInternational Cuisine\nMalaysian Favourites\n48\nAir Batu Campu'


### B4.1 — Register Pathway A

In [11]:
# Register Pathway A — PDF/TXT
register_lineage(
    source=f,
    output=prepared_path,
    processing=approach
)


Registered: IDB_PDF_KLD_EN.pdf -> prepared_text.jsonl


In [12]:
# CELL 6A — GENERATE COMMON LINEAGE MANIFEST
# REQUIRED: Run after all selected pathways are registered
assert lineage_entries, (
    "No lineage entries registered. "
    "Register at least one selected pathway first."
)
lineage_records = []

for i, entry in enumerate(lineage_entries.values(), 1):

    source = entry["source"]
    output = entry["output"]
    processing = entry["processing"]

    # Validate registered files again
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert output.stat().st_size > 0, "Empty output"

    lineage_records.append({
        "source_id": f"S{i:02d}",
        "source_filename": str(
            source.relative_to(RAW_DIR.resolve())
        ),
        "source_sha256": source_hash(source),
        "prepared_output": str(
            output.relative_to(BASE_DIR.resolve())
        ),
        "prepared_sha256": source_hash(output),
        "processing_note": processing,
        "source_version": "Not verified",
        "access_rights": "Not verified",
        "status": "Prepared; quality review pending"
    })

# Reuse the common manifest writer from Cell 4
manifest = save_manifest(lineage_records)

print("\nCOMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(manifest["records"]))
print(
    "Saved to:",
    BASE_DIR / "data" / "week3_manifest.json"
)


Saved lineage: /content/week3_project/data/week3_manifest.json

COMMON LINEAGE MANIFEST GENERATED
Total lineage entries: 1
Saved to: /content/week3_project/data/week3_manifest.json


In [13]:
# CELL 6B — VERIFY COMMON LINEAGE MANIFEST
# REQUIRED: Run after Cell 6A
import json
manifest_path = BASE_DIR / "data" / "week3_manifest.json"
assert manifest_path.is_file(), (
    "Common lineage manifest not found."
)
with manifest_path.open("r", encoding="utf-8") as file:
    saved_manifest = json.load(file)
saved_records = saved_manifest.get("records", [])
assert saved_records, (
    "No lineage records found in saved manifest."
)
assert len(saved_records) == len(lineage_entries), (
    "Saved lineage count does not match registration."
)
for record in saved_records:
    source = RAW_DIR / record["source_filename"]
    output = BASE_DIR / record["prepared_output"]
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert source_hash(source) == record["source_sha256"], (
        f"Source checksum mismatch: {source}"
    )
    assert source_hash(output) == record["prepared_sha256"], (
        f"Prepared output checksum mismatch: {output}"
    )
print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))
print("Manifest:", manifest_path)


COMMON LINEAGE VALIDATION: PASS
Verified records: 1
Manifest: /content/week3_project/data/week3_manifest.json


## Part C — V1 pipeline

In [14]:
# Cell 7 — REQUIRED: Common V1 Pipeline Utilities
# Run ONCE after completing Part B

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

# Step 1: Define standard project paths
BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"

PIPE_DIR.mkdir(parents=True, exist_ok=True)

# Store registered pipeline runs for repeatability verification
PIPELINE_RUNS = {}


# Step 2: Calculate SHA-256 checksum
def sha256_file(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()


# Step 3: Resolve an explicitly selected source
def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path

    if folder:
        if not source.is_dir():
            raise FileNotFoundError(
                f"Source folder not found: {source}"
            )
    else:
        if not source.is_file():
            raise FileNotFoundError(
                f"Source file not found: {source}"
            )

    return source


# Step 4: Execute the selected V1 data pipeline
def execute_pipeline(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    source = Path(source)

    # INGEST: Identify relevant source files
    if kind == "images":
        inputs = sorted(
            p for p in source.iterdir()
            if p.is_file()
            and p.suffix.lower() in {
                ".png", ".jpg", ".jpeg", ".webp"
            }
        )
    else:
        inputs = [source]

    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError(
            "No valid source files to ingest."
        )

    # INSPECT: Apply data-type-specific inspection
    inspection = inspect_fn(inputs, configuration)

    # Define output directory
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)

    # PREPARE / STORE: Apply selected transformation
    outputs, mappings, method = prepare_fn(
        inputs,
        output_dir,
        configuration
    )

    outputs = [Path(p) for p in outputs]

    # VALIDATE: Check prepared outputs
    if not outputs:
        raise ValueError(
            "No prepared outputs were produced."
        )

    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(
                f"Prepared output missing: {output}"
            )

        if output.stat().st_size == 0:
            raise ValueError(
                f"Prepared output is empty: {output}"
            )

    # Record basic validation evidence
    validation = {
        "output_count": len(outputs),
        "output_sha256": [
            sha256_file(p) for p in outputs
        ],
        "output_bytes": [
            p.stat().st_size for p in outputs
        ]
    }

    # RECORD LINEAGE: Create pipeline execution record
    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [
            {
                "path": str(p),
                "sha256": sha256_file(p)
            }
            for p in inputs
        ],
        "outputs": [
            str(p) for p in outputs
        ],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": "Not verified",
        "access_rights": (
            "Not verified — check before sharing"
        )
    }

    # Save pipeline manifest
    manifest_path = output_dir / "pipeline_manifest.json"

    manifest_path.write_text(
        json.dumps(
            record,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print(
        f"SUCCESS: {kind} | "
        f"Inputs: {len(inputs)} | "
        f"Outputs: {len(outputs)}"
    )
    print("Manifest:", manifest_path)

    return record


# Step 5: Register and run a selected pathway
def register_and_run(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    # Keep the runner for repeatability verification
    def runner():
        return execute_pipeline(
            kind,
            source,
            inspect_fn,
            prepare_fn,
            configuration
        )

    first = runner()

    PIPELINE_RUNS[kind] = {
        "first": first,
        "runner": runner
    }
    return first
print("Cell 7 ready.")
print("Run ONLY the selected Cell 8 pathway(s).")
print("Common pipeline utilities are available.")


Cell 7 ready.
Run ONLY the selected Cell 8 pathway(s).
Common pipeline utilities are available.


In [15]:
# Cell 8A — CONDITIONAL REQUIRED: PDF/TXT Projects
# Run only if your project uses document data.
# Requires Cell 7 (Common V1 Pipeline Utilities).

!pip -q install pypdf

import re
import json
from pypdf import PdfReader

# Step 1: Select the source and preparation configuration
# Same source and configuration as the Part B decision.

SOURCE_A = resolve_source(
    "IDB_PDF_KLD_EN.pdf"  # same source as Part B
)

CONFIG_A = {
    "chunk_size": 500,
    "method": "paragraph-aware"
}

# Step 2: Inspect document content
def inspect_documents(inputs, cfg):
    src = inputs[0]

    if src.suffix.lower() == ".pdf":
        reader = PdfReader(str(src))
        pages = [
            page.extract_text() or ""
            for page in reader.pages
        ]
        text = "\n".join(pages)
        page_count = len(pages)

    elif src.suffix.lower() == ".txt":
        text = src.read_text(encoding="utf-8")
        page_count = None

    else:
        raise ValueError("Expected a PDF or TXT source")

    if not text.strip():
        raise ValueError(
            "No extractable text found. "
            "Scanned PDFs may require OCR."
        )

    return {
        "source": src.name,
        "characters": len(text),
        "pages": page_count
    }


# Step 3: Define document preparation
def prepare_documents(inputs, out, cfg):
    src = inputs[0]

    # Extract text
    if src.suffix.lower() == ".pdf":
        reader = PdfReader(str(src))
        text = "\n".join(
            page.extract_text() or ""
            for page in reader.pages
        )

    elif src.suffix.lower() == ".txt":
        text = src.read_text(encoding="utf-8")

    else:
        raise ValueError("Expected a PDF or TXT source")

    # Apply the same cleaning used in Part B
    text = re.sub(r"[ \t]+", " ", text).strip()

    size = int(cfg["chunk_size"])
    method = cfg["method"]

    if size <= 0:
        raise ValueError("Chunk size must be positive")

    # Fixed-size chunking
    if method == "fixed":
        chunks = [
            text[i:i + size]
            for i in range(0, len(text), size)
        ]

    # Paragraph-aware chunking
    elif method == "paragraph-aware":
        chunks = []
        current = ""

        for section in re.split(r"\n\s*\n", text):
            section = section.strip()

            if not section:
                continue

            if len(section) > size:
                if current:
                    chunks.append(current)
                    current = ""

                chunks.extend(
                    section[i:i + size]
                    for i in range(0, len(section), size)
                )

            elif len(current) + len(section) + (2 if current else 0) <= size:
                current += (
                    "\n\n" if current else ""
                ) + section

            else:
                if current:
                    chunks.append(current)

                current = section

        if current:
            chunks.append(current)

    else:
        raise ValueError(
            "Choose 'fixed' or 'paragraph-aware'"
        )

    if not chunks:
        raise ValueError("No text chunks produced")

    # Step 4: Store prepared document records
    target = out / "prepared_text.jsonl"

    with target.open("w", encoding="utf-8") as handle:
        for i, chunk in enumerate(chunks, 1):
            record = {
                "id": i,
                "source": str(src),
                "text": chunk,
                "page": None
            }
            handle.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

    # Step 5: Return source-to-output mapping
    mapping = [
        {
            "source": str(src),
            "output": str(target)
        }
    ]

    processing_method = (
        f"{method} text chunks of {size} characters"
    )

    return [target], mapping, processing_method

# Step 6: Execute and register the pipeline
run_A = register_and_run(
    "documents",
    SOURCE_A,
    inspect_documents,
    prepare_documents,
    CONFIG_A
)
print("Cell 8A completed successfully.")


SUCCESS: documents | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents/pipeline_manifest.json
Cell 8A completed successfully.


Representative prepared record from the pipeline output:

In [16]:
import json
pipe_out = PIPE_DIR / "documents" / "prepared_text.jsonl"
with pipe_out.open(encoding="utf-8") as fh:
    pipe_rows = [json.loads(line) for line in fh if line.strip()]
print("Pipeline records:", len(pipe_rows))
print(json.dumps(pipe_rows[7], ensure_ascii=False)[:400])


Pipeline records: 303
{"id": 8, "source": "/content/week3_project/data/raw/IDB_PDF_KLD_EN.pdf", "text": "th attractions that will surely make your visit a \nbreathtaking experience.\nwww.merdeka-tower.com\nKL Tower\nJalan Puncak, Oﬀ Jalan P.Ramlee\nKL Tower is one of the most prominent and \ndistinctive landmarks in the country. This \ntelecommunications tower, situated atop Bukit \nNanas, is the world’s seventh talles


In [17]:
# Cell 9 — REQUIRED: Repeatability and Provenance Verification
# Run after your selected Cell 8A–8F pathway(s).

# Step 1: Ensure at least one pipeline has been registered
assert PIPELINE_RUNS, (
    "No pipeline runs registered. "
    "Run at least one selected Cell 8 pathway first."
)
print("--- Repeatability Verification ---")
# Step 2: Rerun each selected pipeline
for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]
    # Execute the same pipeline again
    second = entry["runner"]()
    # Step 3: Verify input provenance
    assert first["sources"] == second["sources"], (
        f"Source fingerprint changed for {kind}"
    )
    # Step 4: Verify prepared output fingerprints
    assert first["validation"] == second["validation"], (
        f"Prepared output changed for {kind}"
    )
    # Step 5: Verify source-to-output mapping
    assert (
        first["source_output_mapping"]
        == second["source_output_mapping"]
    ), (
        f"Source-to-output mapping changed for {kind}"
    )
    print(
        f"PASS: {kind} — source fingerprints, "
        "prepared outputs and mappings match"
    )
# Step 6: Display final verification message
print("\nAll selected pipelines passed repeatability smoke checks.")


--- Repeatability Verification ---
SUCCESS: documents | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents/pipeline_manifest.json
PASS: documents — source fingerprints, prepared outputs and mappings match

All selected pipelines passed repeatability smoke checks.


## C4 — Download outputs before Colab disconnects
Packages the prepared outputs and manifests (not the copyrighted source PDFs) into one ZIP.

In [18]:
import shutil
from pathlib import Path
stage = Path("/content/week3_download")
if stage.exists():
    shutil.rmtree(stage)
(stage / "data" / "processed").mkdir(parents=True)
shutil.copy(BASE_DIR / "data" / "processed" / "prepared_text.jsonl", stage / "data" / "processed")
shutil.copy(BASE_DIR / "data" / "week3_manifest.json", stage / "data")
shutil.copytree(BASE_DIR / "data" / "pipeline_v1", stage / "data" / "pipeline_v1")
archive = shutil.make_archive("/content/week3_outputs", "zip", stage)
print("Created:", archive)
for p in sorted(stage.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(stage), p.stat().st_size, "bytes")
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Not in Colab — download", archive, "manually.")


Created: /content/week3_outputs.zip
  data/pipeline_v1/documents/pipeline_manifest.json 1145 bytes
  data/pipeline_v1/documents/prepared_text.jsonl 178360 bytes
  data/processed/prepared_text.jsonl 214828 bytes
  data/week3_manifest.json 629 bytes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>